# Lecture 2: Missing data and test/train splitting  

Filling missing data and test/train spliting are two fundamental techniques of preprocessing. You divide your dataset into two parts: a training set used to fit the model, and a test set used to evaluate the model's performance on unseen data. Missing data are filled after test/train splits and values only from the training set are used.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import TimeSeriesSplit
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

## Handling missing data

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/missing_data_environmental_example.csv")
df["Date"] = pd.to_datetime(df["Date"])
numeric_cols = ['Year', 'Month', 'Air_Temp_degC', 'Humidity_pct',
       'Wind_Speed_m_s', 'Precipitation_mm', 'Cloud_Cover_pct', 'ENSO_Index',
       'Sea_Surface_Temp_degC']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors="coerce")

**Explore the dataset**

Before deciding how to handle missing values, first understand the dataset.
  
For example:  
- How many rows and columns are present?
- Which columns contain missing values?
- Are all numeric-looking columns actually numeric?
- Why might pandas fail to recognize some of our missing values?
- What missing values are there (note: `isna()` will not necessarily identify strings such as `"Unknown"` as missing. Sentinel values such as `99999.0` may also look like valid numbers to pandas.)



In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

display(df.describe(include="all").T)

print("\nMissing values recognized by pandas:")
display(df.isna().sum().sort_values(ascending=False))

**Standardize missing values**

Convert the known missing-value codes to `np.nan`, then convert the environmental variables to numeric.

In [ ]:
missing_codes = [None, "None", "Unknown", "NAN", "NaN", 99999.0, "99999.0"]

clean = df.replace(missing_codes, np.nan).copy()

missing_summary = pd.DataFrame({
    "Missing_n": clean.isna().sum(),
    "Missing_pct": (clean.isna().mean() * 100).round(1)
}).sort_values("Missing_pct", ascending=False)

missing_summary

**Drop missing observations**

The simplest option is to remove rows containing missing data.

This can be reasonable when missingness is rare, but it can substantially reduce the sample size when many rows are affected.

In [ ]:
dropped = clean.dropna()

print("Original rows:", len(clean))
print("Rows after dropna():", len(dropped))
print("Rows removed:", len(clean) - len(dropped))

**Mean, median, and mode imputation**

These methods replace missing observations with a summary statistic.

- **Mean:** useful for approximately symmetric continuous variables.
- **Median:** more resistant to outliers and skewed distributions.
- **Mode:** useful for categorical variables.

In [ ]:
demo = clean.copy()

demo["Air_Temp_mean"] = demo["Air_Temp_degC"].fillna(
    demo["Air_Temp_degC"].mean()
)

**TODO**

Do the same but with median for numeric variables and mode for the catigorical variables.  

In [ ]:
# TODO

**Forward fill (`ffill`)**

Forward fill replaces a missing value with the **most recent previous observation**.

This is common in time series, but it assumes the previous value remains a reasonable estimate.

In [ ]:
ffill_demo = clean.sort_values("Date").copy()

ffill_demo["Air_Temp_ffill"] = ffill_demo["Air_Temp_degC"].ffill()

ffill_demo.loc[ffill_demo["Air_Temp_degC"].isna(),
               ["Date", "Air_Temp_degC", "Air_Temp_ffill"]].head(10)


**Backward fill (`bfill`)**

Backward fill replaces a missing value with the **next available observation**.

Be careful in forecasting applications: `bfill` explicitly uses a future observation and can therefore create data leakage if used improperly.

In [ ]:
bfill_demo = clean.sort_values("Date").copy()

bfill_demo["Air_Temp_bfill"] = bfill_demo["Air_Temp_degC"].bfill()

bfill_demo.loc[bfill_demo["Air_Temp_degC"].isna(),
               ["Date", "Air_Temp_degC", "Air_Temp_bfill"]].head(10)


**Linear interpolation**

Interpolation estimates missing values using neighboring observations. It is often useful for ordered environmental time series with short gaps.

In [ ]:
interp_demo = clean.sort_values("Date").copy()

interp_demo["Air_Temp_interpolated"] = (
    interp_demo["Air_Temp_degC"].interpolate(method="linear")
)

interp_demo.loc[interp_demo["Air_Temp_degC"].isna(),
                ["Date", "Air_Temp_degC", "Air_Temp_interpolated"]].head(10)


**Visual comparison**

The figure below compares observed air temperature with the forward-filled, backward-filled, and interpolated series.

In [ ]:
comparison = clean.sort_values("Date").copy()
comparison["ffill"] = comparison["Air_Temp_degC"].ffill()
comparison["bfill"] = comparison["Air_Temp_degC"].bfill()
comparison["interpolate"] = comparison["Air_Temp_degC"].interpolate()

plt.figure(figsize=(12, 5))
plt.plot(comparison["Date"], comparison["Air_Temp_degC"], marker="o", label="Observed")
plt.plot(comparison["Date"], comparison["ffill"], alpha=0.7, label="Forward fill")
plt.plot(comparison["Date"], comparison["bfill"], alpha=0.7, label="Backward fill")
plt.plot(comparison["Date"], comparison["interpolate"], alpha=0.7, label="Interpolation")
plt.xlabel("Date")
plt.ylabel("Air temperature (°C)")
plt.legend()
plt.show()

**Group-based imputation**

Environmental data often have known structure. For monthly climate data, a missing July temperature may be better estimated from other Julys than from the overall annual median.

Here we fill missing air temperatures using the median for the corresponding calendar month.


In [ ]:
group_demo = clean.copy()

group_demo["Air_Temp_month_median"] = group_demo["Air_Temp_degC"].fillna(
    group_demo.groupby("Month")["Air_Temp_degC"].transform("median")
)

group_demo.loc[group_demo["Air_Temp_degC"].isna(),
               ["Date", "Month", "Air_Temp_degC", "Air_Temp_month_median"]].head(10)


**`SimpleImputer` in scikit-learn**

Scikit-learn provides preprocessing tools that make imputation easier to integrate into a machine-learning workflow.


In [ ]:
features = [
    "Year", "Month", "Air_Temp_degC", "Humidity_pct",
    "Wind_Speed_m_s", "Precipitation_mm",
    "Cloud_Cover_pct", "ENSO_Index"
]

model_data = clean.dropna(subset=["Sea_Surface_Temp_degC"]).sort_values("Date")

X = model_data[features]
y = model_data["Sea_Surface_Temp_degC"]

# Chronological holdout: do NOT shuffle a forecasting dataset.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, shuffle=False
)

median_imputer = SimpleImputer(strategy="median")

X_train_median = median_imputer.fit_transform(X_train)
X_test_median = median_imputer.transform(X_test)

print("Missing values in original training set:", X_train.isna().sum().sum())
print("Missing values after imputation:", np.isnan(X_train_median).sum())

## Test/train splitting
The `train_test_split` function from scikit-learn (sklearn) automates this process, handling the random shuffling and splitting with just one line of code. The simplest usage of `train_test_split` requires just two arguments: your features and your target variable.

In [ ]:
# Load sample data
model_data = clean.dropna(subset=["Sea_Surface_Temp_degC"]).sort_values("Date").reset_index(drop=True)

X = model_data[features]
y = model_data["Sea_Surface_Temp_degC"]
 
# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

This splits your data randomly, with 80% going to training and 20% to testing. However, this basic usage has a critical flaw: the split is different every time you run the code, making results irreproducible.  
  
**TODO**: Change the above code to split into 70% training and 30% testing.

In [ ]:
#TODO

Note `test_size` can also be an integer if you want to specify the exact number of samples.

**Reproducibility**  

The `random_state` parameter is crucial for reproducible results. Without it, you get a different split every time you run your code, making it impossible to debug or compare experiments.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print("Testing set #1:", y_test[:10].values)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print("Testing set #2:", y_test[:10].values)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)
print("Testing set #3 (random state=4):", y_test[:10].values)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)
print("Testing set #4 (random state=4):", y_test[:10].values)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=5)
print("Testing set #5 (random state=5):", y_test[:10].values)

By default, `train_test_split` shuffles the data before splitting. For most machine learning tasks, this is exactly what you want. However, for time series data or when the order matters, you should disable shuffling.

In [ ]:
# Shuffle enabled (default)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True)
print("Testing set #1:", y_test[:10])

# Shuffle disabled (for time series)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
print("Testing set #2:", y_test[:10])

**Time Series Splitting**

For time series data, random shuffling destroys the temporal order, which can lead to data leakage (using future information to predict the past). Use `TimeSeriesSplit` instead:

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
 
for fold, (train_index, test_index) in enumerate(tscv.split(X), start=1):

    X_train = X.iloc[train_index]
    X_test = X.iloc[test_index]

    y_train = y.iloc[train_index]
    y_test = y.iloc[test_index]

    print(f"Fold {fold}")
    print(
        model_data.loc[train_index, "Date"].min(),
        "to",
        model_data.loc[train_index, "Date"].max()
    )
    print(
        model_data.loc[test_index, "Date"].min(),
        "to",
        model_data.loc[test_index, "Date"].max()
    )
    print()

**Stratified Splitting for Imbalanced Data**  

When your dataset has imbalanced classes (some classes have far fewer samples than others), random splitting can create training or test sets that poorly represent the overall distribution. This is especially problematic for classification tasks.

The stratify parameter ensures that the class distribution in the training and test sets matches the original dataset.

In [ ]:
# Create imbalanced dataset (95% class 0, 5% class 1)
X = np.random.randn(1000, 5)
y = np.array([0] * 950 + [1] * 50)
 
# Without stratification
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=71)
print(f"Train distribution: Class 0: {sum(y_train == 0)}, Class 1: {sum(y_train == 1)}, Class Ratio: {sum(y_train == 1)/sum(y_train == 0)}")
print(f"Test distribution: Class 0: {sum(y_test == 0)}, Class 1: {sum(y_test == 1)}, Class Ratio: {sum(y_train == 1)/sum(y_train == 0)}")
 
# With stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=71, stratify=y
)
print(f"\nStratified train distribution: Class 0: {sum(y_train == 0)}, Class 1: {sum(y_train == 1)}, Class Ratio: {sum(y_train == 1)/sum(y_train == 0)}")
print(f"Stratified test distribution: Class 0: {sum(y_test == 0)}, Class 1: {sum(y_test == 1)}, Class Ratio: {sum(y_train == 1)/sum(y_train == 0)}")

## Group Problem

In [ ]:
import json
import requests

url = "https://raw.githubusercontent.com/dblaskey/ML_Course_Code/main/Data/Climate.json"

# Perform a network request to fetch the data
response = requests.get(url)

# Convert the response content directly into a Python dictionary
data = response.json()
data

In [ ]:
parquet_url = data["data_set_description"]["parquet_url"]
df = pd.read_parquet(parquet_url)

We want to predict `Sea_Surface_Temp_degC` as a function of the other columns in the dataframe.

**TODO**:  
1. Explore the dataset
2. Remove `Latitude`, `Longitude`, and `Altitude_m` columns
3. Change all `None`, `Unknown`, `NAN`, `NaN`, and `99999.0` to `np.nan`
4. Fill all missing values with the most appropriate method
5. If the goal is to create a model to predict future sea surface temperatures given new data, split the dataset into appropriate test and train sets.

In [ ]:
#TODO